<a href="https://colab.research.google.com/github/Scrapjon/Face-and-Emotion-Detection/blob/main/src/face_emotion/face_recognition/VGGFace_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine Tuning of VGGFace for Facial Recognition

In [1]:
import time
import sys
import gdown
import os
from typing import List, cast, Any
from numpy.typing import NDArray
#
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Convolution2D,
                                     ZeroPadding2D,
                                     MaxPooling2D,
                                     Flatten,
                                     Dropout,
                                     Activation)

## Load Base Model
* `base_model()` constructs sequential model
* `load_model()` then applies the weights from the provided url
* `model` the base model with the applied weights

In [2]:
BASE_MODEL_WEIGHTS_URL = (
    "https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5"
)

In [3]:
def base_model() -> Sequential:
  model = Sequential()
  model.add(ZeroPadding2D((1, 1), input_shape=(224, 224, 3)))
  model.add(Convolution2D(64, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(64, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(128, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(128, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(256, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(256, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(256, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(Convolution2D(4096, (7, 7), activation="relu"))
  model.add(Dropout(0.5))
  model.add(Convolution2D(4096, (1, 1), activation="relu"))
  model.add(Dropout(0.5))
  model.add(Convolution2D(2622, (1, 1)))
  model.add(Flatten())
  model.add(Activation("softmax"))
  return model

In [9]:
def load_model(
  url: str = BASE_MODEL_WEIGHTS_URL
) -> Model:
  # construct model
  model: Model = base_model()
  # check if file already downloaded
  def download_weights_if_necessary(
      file_name: str = "base_weights.h5",
      url:str = url
    ) -> os.path:
    target_file = os.path.normpath(f"vggFineTuning/{file_name}")
    if not os.path.isdir('vggFineTuning'):
      os.mkdir('vggFineTuning')
    if os.path.isfile(target_file):
      print(f"weights already downloaded at {target_file}")
      return target_file
    #
    try:
      print(f"{file_name} will be downloaded from {url} to {target_file}")
      gdown.download(BASE_MODEL_WEIGHTS_URL, target_file, quiet=False)
    except Exception as e:
      raise ValueError(
        f"Something has gone while downloading {file_name} from {url}"
      ) from e
    # potentially add downloading/uncompressing of .zip/.bz2 files?
    return target_file
  weight_file = download_weights_if_necessary()
  try:
    model.load_weights(weight_file)
  except Exception as e:
    raise ValueError(
      f"Something has gone loading pre-trained weights from {weight_file}"
    ) from e
  # model here will be 2622d dimensions
  base_model_output = Flatten()(model.layers[-5].output)
  # flatten to 4096 dimensions
  # apparently increases accuracy according to the comments in the DeepFace code
  vgg_face_descriptor = Model(inputs=model.layers[0].input, outputs=base_model_output)
  return vgg_face_descriptor


In [10]:
try:
  model = load_model(url=BASE_MODEL_WEIGHTS_URL)
  model.summary()
except Exception as e:
  import traceback
  traceback.print_exc()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/zero_padding2d.py:72: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


base_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to vggFineTuning/base_weights.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /content/vggFineTuning/base_weights.h5
100%|██████████| 580M/580M [00:17<00:00, 33.5MB/s]


Model: "functional_152"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_39               │ (None, 226, 226, 3)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_48 (Conv2D)              │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_40               │ (None, 226, 226, 64)   │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_49 (Conv2D)              │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_41               │ (None, 114, 114, 64)   │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_50 (Conv2D)              │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_42               │ (None, 114, 114, 128)  │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_51 (Conv2D)              │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_43               │ (None, 58, 58, 128)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_52 (Conv2D)              │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_44               │ (None, 58, 58, 256)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_53 (Conv2D)              │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_45               │ (None, 58, 58, 256)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_54 (Conv2D)              │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_46               │ (None, 30, 30, 256)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_55 (Conv2D)              │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_47               │ (None, 30, 30, 512)    │             

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 134,260,544 (512.16 MB)

 Non-trainable params: 0 (0.00 B)

## Download & Preparing Dataset
[Link to Dataset](https://www.kaggle.com/competitions/11-785-fall-20-homework-2-part-2/data)

In [17]:
import kagglehub
# API token (do not steal):
#KGAT_4bd8e8f794e814c9efea224d573e0f44
print("vvv this will appear even if you're already logged in <3")
kagglehub.login()
path = kagglehub.competition_download("11-785-fall-20-homework-2-part-2")
print(f"dataset downloaded at {path}")
test_dir  = path + "/classification_data/test_data"
train_dir = path + "/classification_data/train_data"
val_dir   = path + "/classification_data/val_data"
print(f"testing data directory: {test_dir}")
print(f"training data directory: {train_dir}")
print(f"validation data directory: {val_dir}")

vvv this will appear even if you're already logged in <3


dataset downloaded at /root/.cache/kagglehub/competitions/11-785-fall-20-homework-2-part-2
testing data directory: /root/.cache/kagglehub/competitions/11-785-fall-20-homework-2-part-2/classification_data/test_data
training data directory: /root/.cache/kagglehub/competitions/11-785-fall-20-homework-2-part-2/classification_data/train_data
validation data directory: /root/.cache/kagglehub/competitions/11-785-fall-20-homework-2-part-2/classification_data/val_data


In [24]:
from tensorflow.keras.utils import image_dataset_from_directory
#
# model input shape is (None, 224, 224, 3)
# > 224x224 image, 3 colour channels
batch_size = 32
image_size = (224, 224)
#
train_ds = image_dataset_from_directory(
  directory = train_dir,
  image_size = image_size,
  batch_size = batch_size
)
val_ds = image_dataset_from_directory(
  directory = val_dir,
  image_size = image_size,
  batch_size = batch_size
)
test_ds = image_dataset_from_directory(
  directory = test_dir,
  image_size = image_size,
  batch_size = batch_size
)

Found 380638 files belonging to 4000 classes.
Found 8000 files belonging to 4000 classes.
Found 8000 files belonging to 4000 classes.


## Fine Tuning Time
we have a large dataset; therefore we should be able to get full fine tuning.
If I have time may extend to potentially also have the option to only fine tune the final layers

In [16]:
model.trainable = True
model.compile(
  optimizer=tensorflow.keras.optimizers.Adam(1e-5),
  loss=tensorflow.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
  metrics=[tensorflow.keras.metrics.SparseCategoricalAccuracy('accuracy')]
)
#
num_epochs = 10
history = model.fit(
  train_ds,
  epochs=num_epochs,
  validation_data=val_ds
)

36


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val']
#
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.ylim([0.8, 1])
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.ylim([0, 1.0])
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()